# Evaluate conditionally generated molecules

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from plotly import express as px
from tqdm import tqdm

In [ ]:
from tools import compute_uniqueness, compute_novelty, compute_unique_novelty

## Load data

In [ ]:
pred_dir = Path("predictions/conditional_mol/attempt_2")
files = sorted(f for f in pred_dir.glob("*.csv") if not f.stem.endswith("conditional"))
dfs = []
for file in files:
    print(file)
    df = pd.read_csv(file, low_memory=False).reset_index()
    # Evaluation script does not break molecules correctly, introducting these rows
    df = df[~df["fail"].fillna(0).astype(bool)].reset_index()
    df = df.drop(columns=["index", "fail"])

    df_cond = pd.read_csv(
        Path(file).parent / (Path(file).stem + "_conditional.csv"), low_memory=False
    ).reset_index()
    df_cond.columns = [c.lower().replace(" ", "_") for c in df_cond.columns]
    df_cond = df_cond.drop(columns=["index", "fail", "error"], errors="ignore")

    if len(df) != len(df_cond):
        print(f"Lengths different: {len(df)} != {len(df_cond)}")
        continue
    df = pd.concat([df, df_cond], axis=1, ignore_index=False)

    df["method"] = Path(file).stem
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

df["total"] = True
df["valid"] = df["connected"] & df["chemical"] & df["physical"]

## Load references

In [ ]:
file = "/homes/buttensc/Projects/semla-flow/data/conditional_mol/test_first_1000.csv"
truth = pd.read_csv(file)
truth = truth[truth.fail != 1.0]
print(len(truth))

# Table: validity

In [ ]:
cols = ["method", "total", "connected", "chemical", "physical", "valid"]

df_valid = df[cols].groupby("method").sum()
df_valid.index.name = None
print(df_valid)

df_valid = df[cols].groupby("method").mean()
df_valid.index.name = None
df_valid.style.format("{:.2%}")

# Table: scaffold

In [ ]:
cols = ["method", "scaffold_true_csk", "scaffold_rdkit_csk"]
df_scaff = df[df.valid][cols].groupby("method").mean()
df_scaff.style.format("{:.2%}")

In [ ]:
metrics = {
    "sucos": "SuCOS",
    "tanimoto": "ECFP4 Bit Tanimoto",
    "ensemble_avg_energy": "Ensemble Average Energy",
    "mol_pred_energy": "Molecular Prediction Energy",
    "energy_ratio": "Energy Ratio",
    "sa": "Synthetic Accessability Score",
    "sa_normalized": "Synthetic Accessability Score (normalized)",
    "spacial": "Spacial Score",
    "qed": "QED",
    "logp": "LogP",
    "lipinski": "Lipinski Rule of 5",
    "num_heavy": "Number of Heavy Atoms",
    "weight": "Molecular Weight",
    "num_rings": "Number of Rings",
}

## Show

In [ ]:
metric = "sucos"
# metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    # cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

In [ ]:
# metric = "sucos"
metric = "tanimoto"
name = metrics[metric]
sns.histplot(
    df[["method", metric]].reset_index(drop=True),
    x=metric,
    hue="method",
    bins=100,
    cumulative=True,
    common_norm=False,
    stat="density",
    element="step",
    # kde=True,
    fill=False,
    # legend=True, palette="tab10", linewidth=1.5
)

# Novelty and uniqueness

In [ ]:
# df_train = pd.read_csv("/homes/buttensc/Projects/semla-flow/data/unconditional/geom-drugs/train.csv")
# df_train["method"] = "GEOM Drugs Training"

df_test = pd.read_csv(
    "/homes/buttensc/Projects/semla-flow/data/unconditional/geom-drugs/test.csv"
)
df_test["method"] = "GEOM Drugs Testing"

comparison_smiles = set(df_test["smiles"].dropna()) - {None, "", pd.NA, np.nan}

In [ ]:
def compute_uniqueness(smiles: list[str]) -> float:
    """Compute the uniqueness of a list of SMILES strings."""
    valid_smiles = [s for s in smiles if s not in {None, "", pd.NA, np.nan}]  # list
    return len(set(valid_smiles)) / len(valid_smiles)


def compute_novelty(
    smiles: list[str], reference_smiles: set[str] = comparison_smiles
) -> float:
    """How many are not in the test set?"""
    # valid_smiles = set(s for s in smiles if s not in {None, "", pd.NA, np.nan})  # set
    valid_smiles = list(s for s in smiles if s not in {None, "", pd.NA, np.nan})  # list
    return len(
        [smiles for smiles in valid_smiles if smiles not in reference_smiles]
    ) / len(valid_smiles)


def compute_unique_novelty(
    smiles: list[str], reference_smiles: set[str] = comparison_smiles
) -> float:
    """How many unique new molecules have we generated?"""
    # valid_smiles = set(s for s in smiles if s not in {None, "", pd.NA, np.nan})  # set
    valid_smiles = list(s for s in smiles if s not in {None, "", pd.NA, np.nan})  # list
    return len(set(valid_smiles) - reference_smiles) / len(valid_smiles)


In [ ]:
# How much repetition is there? How unique are the generated molecules?
s = df.groupby("method")["smiles_pred"].agg(compute_uniqueness)
s.name = "Uniqueness"
s

In [ ]:
# How many of the valid generated molecules are not in the test set?
s = df.groupby("method")["smiles_pred"].agg(compute_novelty)
s.name = "Novelty"
s

In [ ]:
# How many of the valid generated molecules are in the test set?
s = (df.groupby("method")["smiles_pred"].agg(compute_novelty) - 1).abs()
s.name = "In Test Set"
s

In [ ]:
# How many valid, unique and new molecules have we generated?
s = df.groupby("method")["smiles_pred"].agg(compute_unique_novelty)
s.name = "Unique Novelty"
s